# MOD-02 · NB-05 — Missingness Analysis & Imputation
### Health Analytics with Python · Module 02: EDA in Healthcare
---
**Learning objectives**
- Visualise missing data patterns with `missingno`
- Test MCAR, MAR, and MNAR mechanisms with statistical tests
- Apply median, LOCF, and multiple imputation (MICE) strategies
- Compare imputed vs original distributions with overlapping KDE plots
- Document imputation decisions for a reproducible analysis plan

**Estimated time:** 2 hours | **Level:** Intermediate | **Libraries:** `pandas`, `numpy`, `missingno`, `sklearn`, `matplotlib`


## 1. Setup & Dataset with Realistic Missingness

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, ttest_ind
import warnings
warnings.filterwarnings('ignore')

try:
    import missingno as msno
    print("missingno imported successfully.")
except ImportError:
    print("Run: pip install missingno")

from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.experimental import enable_iterative_imputer  # noqa

sns.set_theme(style='whitegrid', font_scale=1.02)
plt.rcParams.update({'figure.dpi':120,'figure.facecolor':'white'})

# ── Base dataset ──────────────────────────────────────────────────────────────
np.random.seed(42); N=800
ages      = np.random.normal(62,16,N).clip(18,95).astype(int)
sexes     = np.random.choice(['M','F'],N,p=[0.48,0.52])
payers    = np.random.choice(['Medicare','Medicaid','Private','Self-pay','Other'],N,p=[0.40,0.22,0.28,0.07,0.03])
los_days  = np.random.gamma(2.5,1.8,N).clip(1,30).astype(int)
charges   = (los_days*np.random.uniform(1800,4200,N)).round(2)
np.random.seed(99)
df = pd.DataFrame({
    'patient_id': [f'PT-{str(i).zfill(5)}' for i in range(1,N+1)],
    'age':ages,'sex':sexes,'payer':payers,'los_days':los_days,
    'diabetes':np.random.binomial(1,0.32,N),
    'hypertension':np.random.binomial(1,0.45,N),
    'readmitted_30':np.random.binomial(1,0.14,N),
    'total_charge':charges,
})

# ── Introduce structured missingness (realistic clinical patterns) ─────────────
np.random.seed(7)

# MCAR — completely random (8% glucose)
df['glucose'] = np.random.normal(105,28,N).clip(50,400).round(1)
df.loc[np.random.rand(N)<0.08, 'glucose'] = np.nan

# MAR — missing more often in Self-pay (lab deferral due to cost)
df['creatinine'] = np.random.gamma(1.6,0.6,N).clip(0.4,10).round(2)
selfpay_mask = (df['payer']=='Self-pay')
df.loc[selfpay_mask & (np.random.rand(N)<0.55), 'creatinine'] = np.nan
df.loc[~selfpay_mask & (np.random.rand(N)<0.10), 'creatinine'] = np.nan

# MNAR — HbA1c missing more often when values are in normal range (not ordered)
df['hba1c'] = np.random.normal(6.8,1.4,N).clip(4,14).round(1)
low_hba1c = df['hba1c'] < 5.7
df.loc[low_hba1c & (np.random.rand(N)<0.60), 'hba1c'] = np.nan
df.loc[~low_hba1c & (np.random.rand(N)<0.15), 'hba1c'] = np.nan

# Charge missing correlated with Self-pay
df['total_charge'] = charges
df.loc[selfpay_mask & (np.random.rand(N)<0.40), 'total_charge'] = np.nan
df.loc[~selfpay_mask & (np.random.rand(N)<0.05), 'total_charge'] = np.nan

print(f"Dataset: {df.shape}")
missing_pct = df.isnull().mean()*100
print("\nMissingness summary:")
display(missing_pct[missing_pct>0].round(1).to_frame('% missing'))


## 2. Visualising Missingness with missingno

In [ ]:
lab_cols = ['glucose','creatinine','hba1c','total_charge','age','los_days']
df_vis   = df[lab_cols].copy()

fig, axes = plt.subplots(1,2,figsize=(15,5))

# Bar chart — % present per variable
msno.bar(df_vis, ax=axes[0], color='#4878CF', fontsize=11)
axes[0].set_title('Missingness bar chart — % data present')

# Matrix — co-occurrence of missing values
msno.matrix(df_vis, ax=axes[1], color=(0.3,0.5,0.7), fontsize=11, sparkline=False)
axes[1].set_title('Missingness matrix — co-occurrence patterns')

plt.tight_layout()
plt.savefig('/tmp/mod02/missingness_bar_matrix.png', bbox_inches='tight')
plt.show()


In [ ]:
# Heatmap — correlation of missingness between variables
fig, ax = plt.subplots(figsize=(7,5))
msno.heatmap(df_vis, ax=ax, fontsize=11, cmap='RdBu_r')
ax.set_title('Missingness correlation heatmap')
plt.tight_layout()
plt.savefig('/tmp/mod02/missingness_heatmap.png', bbox_inches='tight')
plt.show()
print("High positive correlation → variables tend to be missing TOGETHER")
print("High negative correlation → variables missing in opposite rows")


## 3. Testing Missing Data Mechanisms

In [ ]:
print("=" * 60)
print("Testing Missing Data Mechanisms")
print("=" * 60)

# ── MCAR test for Glucose — is missingness related to observed variables? ─────
print("\n--- Glucose (expected: MCAR) ---")
df['glucose_miss'] = df['glucose'].isnull().astype(int)

# t-test: is mean age different in missing vs observed rows?
g0 = df.loc[df['glucose_miss']==0,'age']
g1 = df.loc[df['glucose_miss']==1,'age']
t, p = ttest_ind(g0,g1)
print(f"Age difference test: t={t:.3f}, p={p:.3f} {'← age predicts missingness (not MCAR)' if p<0.05 else '← no age effect (consistent with MCAR)'}")

ct  = pd.crosstab(df['payer'], df['glucose_miss'])
chi2, p_chi, _, _ = chi2_contingency(ct)
print(f"Payer association: χ²={chi2:.2f}, p={p_chi:.3f} {'← MAR by payer' if p_chi<0.05 else '← no payer effect'}")

# ── MAR test for Creatinine ────────────────────────────────────────────────────
print("\n--- Creatinine (expected: MAR by payer) ---")
df['creat_miss'] = df['creatinine'].isnull().astype(int)
ct2 = pd.crosstab(df['payer'], df['creat_miss'])
chi2_c, p_c, _, _ = chi2_contingency(ct2)
print(f"Payer association: χ²={chi2_c:.2f}, p={p_c:.4f} {'← MAR confirmed' if p_c<0.05 else ''}")
print("Missing rates by payer:")
display(df.groupby('payer')['creat_miss'].mean().round(3).to_frame('prop_missing'))

# ── MNAR — show HbA1c missing rate differs by true value range ────────────────
print("\n--- HbA1c (expected: MNAR — missing when low) ---")
print("Note: For MNAR, we cannot fully test with observed data alone.")
print("Pattern check — look at which ranges are missing most:")
# Use diabetes flag as a proxy for true HbA1c range
miss_diab = df.groupby('diabetes')['hba1c'].apply(lambda x: x.isnull().mean())
print(miss_diab.rename('prop_hba1c_missing'))
print("(If non-diabetic patients have higher missingness, MNAR is plausible)")


## 4. Imputation Strategies

In [ ]:
# Store complete-case subsets for comparison
glucose_complete = df['glucose'].dropna().values

# ── 4.1  Median imputation (simple, robust to outliers) ─────────────────────
df['glucose_median'] = df['glucose'].fillna(df['glucose'].median())
print(f"Median imputed: {df['glucose'].isnull().sum()} NaN → 0")

# ── 4.2  LOCF — Last Observation Carried Forward (longitudinal data) ─────────
# Simulate a longitudinal HbA1c series for 50 patients over 4 quarters
np.random.seed(55)
n_pts, n_periods = 50, 4
long_df = pd.DataFrame({
    'patient_id': np.repeat([f'PT-{i:03d}' for i in range(n_pts)], n_periods),
    'quarter'   : list(range(n_periods)) * n_pts,
    'hba1c'     : np.random.normal(6.8,1.2, n_pts*n_periods).clip(4,13).round(1)
})
# Introduce 25% missingness
long_df.loc[np.random.rand(len(long_df))<0.25,'hba1c'] = np.nan
long_df['hba1c_locf'] = (long_df.groupby('patient_id')['hba1c']
                                  .ffill())
n_still_missing = long_df['hba1c_locf'].isnull().sum()
print(f"\nLOCF — residual missing after forward fill: {n_still_missing}")
print("(Residual = rows where patient has no prior observation)")


In [ ]:
# ── 4.3  KNN Imputation ───────────────────────────────────────────────────────
num_cols = ['age','los_days','glucose','creatinine','hba1c','total_charge']
df_num   = df[num_cols].copy()

knn_imp = KNNImputer(n_neighbors=5, weights='distance')
df_knn  = pd.DataFrame(knn_imp.fit_transform(df_num), columns=num_cols)
print("KNN imputation complete.")
print(f"Missing before: {df_num.isnull().sum().sum()}")
print(f"Missing after : {df_knn.isnull().sum().sum()}")


In [ ]:
# ── 4.4  MICE — Multiple Imputation by Chained Equations ────────────────────
mice_imp = IterativeImputer(
    random_state=42,
    max_iter=10,
    n_nearest_features=5,
    initial_strategy='median'
)
df_mice = pd.DataFrame(mice_imp.fit_transform(df_num), columns=num_cols)
print("MICE imputation complete.")
print(f"Missing after MICE: {df_mice.isnull().sum().sum()}")


## 5. Comparing Imputed Distributions

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(15,4))
fig.suptitle('Distribution comparison — original vs imputation strategies', fontsize=12)

for ax, col, col_knn, col_mice, title in zip(
        axes,
        ['glucose','creatinine','hba1c'],
        ['glucose','creatinine','hba1c'],
        ['glucose','creatinine','hba1c'],
        ['Glucose (mg/dL)','Creatinine (mg/dL)','HbA1c (%)']):
    original = df[col].dropna()
    knn_vals = df_knn[col_knn]
    mice_vals= df_mice[col_mice]

    sns.kdeplot(original,  ax=ax, label='Original (complete cases)', color='black',  lw=2)
    sns.kdeplot(knn_vals,  ax=ax, label='KNN imputed',  color='#4878CF', lw=1.5, ls='--')
    sns.kdeplot(mice_vals, ax=ax, label='MICE imputed', color='#D65F5F', lw=1.5, ls=':')

    ax.set_title(title); ax.set_ylabel('Density'); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('/tmp/mod02/imputation_comparison.png', bbox_inches='tight')
plt.show()
print("Good imputation: KDE of imputed values closely follows original.")
print("Warning signs: bimodal bumps, shifted peaks, or flattened tails.")


## 6. Imputation Decision Framework

In [ ]:
decision_rules = {
    'glucose'     : {'mechanism':'MCAR', 'pct_missing':8,
                     'recommendation':'Median or KNN — any method acceptable',
                     'chosen':'KNN (preserves multivariate relationships)'},
    'creatinine'  : {'mechanism':'MAR (by payer)', 'pct_missing':18,
                     'recommendation':'MICE — can use payer as auxiliary variable',
                     'chosen':'MICE with payer as predictor'},
    'hba1c'       : {'mechanism':'MNAR (missing when normal)', 'pct_missing':35,
                     'recommendation':'Sensitivity analysis required — imputation may bias results',
                     'chosen':'MICE + sensitivity analysis with worst-case scenario'},
    'total_charge': {'mechanism':'MAR (by payer)', 'pct_missing':12,
                     'recommendation':'MICE or KNN; log-transform before imputing',
                     'chosen':'MICE on log(charge), back-transform to original scale'},
}

print(f"{'Variable':15s} {'Mechanism':25s} {'%miss':>6s}  Chosen strategy")
print("-"*85)
for var, d in decision_rules.items():
    print(f"{var:15s} {d['mechanism']:25s} {d['pct_missing']:>5d}%  {d['chosen']}")


## 7. Knowledge Check
1. What is the key difference between MCAR and MAR? Why does it matter for imputation?
2. Why should you NOT use LOCF for a cross-sectional dataset?
3. MICE ran 10 iterations. How would you check if it converged?
4. HbA1c has 35% MNAR missingness. Why might any imputation method produce biased estimates?
5. Re-run KNN imputation with `n_neighbors=10`. Do the distributions change materially?


In [ ]:
# Exercise 5 — KNN with k=10
knn10 = KNNImputer(n_neighbors=10, weights='distance')
df_knn10 = pd.DataFrame(knn10.fit_transform(df_num), columns=num_cols)

fig, ax = plt.subplots(figsize=(7,4))
sns.kdeplot(df['glucose'].dropna(),  ax=ax, label='Original', color='black', lw=2)
sns.kdeplot(df_knn['glucose'],       ax=ax, label='KNN k=5',  color='#4878CF', ls='--')
sns.kdeplot(df_knn10['glucose'],     ax=ax, label='KNN k=10', color='#D65F5F', ls=':')
ax.set_title('Glucose: KNN k=5 vs k=10'); ax.legend(); plt.show()


---
**Next → NB-06: Comorbidity Co-occurrence Heatmap**